<a href="https://colab.research.google.com/github/hyunkyung31/coronary-ai-ml-dl/blob/main/hyunkyung/02_AngioCAD_%EC%A0%84%EC%B2%98%EB%A6%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. 데이터 및 감사 결과 복구
2. Clinical cleaning rule 재적용
3. 이미지 pixel hash 중복 검사

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

!apt-get update -qq
!apt-get install -y -qq unrar

Mounted at /content/drive
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
#@title 프로젝트 경로 자동 탐색

from pathlib import Path
import pandas as pd
import numpy as np

DRIVE_ROOT = Path("/content/drive/MyDrive")

part1_candidates = list(
    DRIVE_ROOT.rglob(
        "AngioCAD_Dataset.part1.rar"
    )
)

if len(part1_candidates) != 1:
    raise RuntimeError(
        f"part1.rar 검색 결과: "
        f"{len(part1_candidates)}개"
    )

PROJECT_DIR = part1_candidates[0].parent
AUDIT_DIR = PROJECT_DIR / "audit_results"
PREPROCESS_DIR = PROJECT_DIR / "preprocessing_results"
SPLIT_DIR = PROJECT_DIR / "splits"

PREPROCESS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SPLIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

EXTRACT_ROOT = Path("/content/AngioCAD")
DATASET_DIR = (
    EXTRACT_ROOT / "AngioCAD_Dataset"
)

FEATURES_FILE = (
    PROJECT_DIR / "AngioCAD_Features.xlsx"
)

print("프로젝트:", PROJECT_DIR)
print("Audit:", AUDIT_DIR)
print("전처리 결과:", PREPROCESS_DIR)
print("Split:", SPLIT_DIR)

프로젝트: /content/drive/MyDrive/[MacGyver]최종프로젝트/03_data/AngioCAD
Audit: /content/drive/MyDrive/[MacGyver]최종프로젝트/03_data/AngioCAD/audit_results
전처리 결과: /content/drive/MyDrive/[MacGyver]최종프로젝트/03_data/AngioCAD/preprocessing_results
Split: /content/drive/MyDrive/[MacGyver]최종프로젝트/03_data/AngioCAD/splits


In [ ]:
#@title 필수 감사 결과 존재 여부

REQUIRED_AUDIT_FILES = [
    "13_series_view_manifest.csv",
    "14_paper_model_series.csv",
    "15_extended_model_series.csv",
    "19_patient_label_summary.csv",
    "20_paper_series_with_labels.csv",
    "21_extended_series_with_labels.csv",
    "28_patient_modality_eligibility.csv",
    "36_clinical_cleaning_rules.csv",
    "37_clinical_cleaned_preview.csv",
]

missing_audit_files = [
    file_name
    for file_name in REQUIRED_AUDIT_FILES
    if not (
        AUDIT_DIR / file_name
    ).exists()
]

if missing_audit_files:
    raise FileNotFoundError(
        "필수 감사 결과 누락:\n"
        + "\n".join(missing_audit_files)
    )

print("✅ 필수 감사 결과가 모두 존재합니다.")

✅ 필수 감사 결과가 모두 존재합니다.


In [ ]:
#@title rar 재압축 해제
# 분할 압축 및 디스크 공간 확인

import shutil

rar_parts = [
    PROJECT_DIR
    / f"AngioCAD_Dataset.part{i}.rar"
    for i in range(1, 5)
]

for path in rar_parts:
    print(
        path.name,
        "존재:",
        path.exists(),
        "크기:",
        (
            f"{path.stat().st_size / 1024**3:.2f} GiB"
            if path.exists()
            else "-"
        ),
    )

assert all(
    path.exists()
    for path in rar_parts
), "분할압축 파일이 누락됐습니다."

total, used, free = shutil.disk_usage(
    "/content"
)

print(
    f"\n남은 로컬 공간: "
    f"{free / 1024**3:.1f} GiB"
)

if free < 25 * 1024**3:
    raise RuntimeError(
        "로컬 여유 공간이 25GiB 미만입니다."
    )


AngioCAD_Dataset.part1.rar 존재: True 크기: 3.91 GiB
AngioCAD_Dataset.part2.rar 존재: True 크기: 3.91 GiB
AngioCAD_Dataset.part3.rar 존재: True 크기: 3.91 GiB
AngioCAD_Dataset.part4.rar 존재: True 크기: 3.56 GiB

남은 로컬 공간: 205.5 GiB


In [ ]:
#@title 필요한 경우에만 압축 해제

import subprocess
import time

if DATASET_DIR.exists():
    existing_patients = [
        path
        for path in DATASET_DIR.iterdir()
        if path.is_dir()
        and path.name.isdigit()
    ]
else:
    existing_patients = []

if len(existing_patients) == 413:
    print(
        "기존 압축 해제본을 재사용합니다."
    )
else:
    EXTRACT_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    start_time = time.time()

    command = [
        "unrar",
        "x",
        "-o+",
        "-idq",
        str(rar_parts[0]),
        str(EXTRACT_ROOT) + "/",
    ]

    print("압축 해제를 시작합니다.")

    result = subprocess.run(command)

    if result.returncode != 0:
        raise RuntimeError(
            f"압축 해제 실패: "
            f"code={result.returncode}"
        )

    print(
        f"압축 해제 완료: "
        f"{(time.time()-start_time)/60:.1f}분"
    )

압축 해제를 시작합니다.
압축 해제 완료: 6.5분


In [ ]:
#@title 압축해제 결과 재검증

patient_dirs = sorted(
    [
        path
        for path in DATASET_DIR.iterdir()
        if path.is_dir()
        and path.name.isdigit()
    ],
    key=lambda path: int(path.name),
)

series_dirs = [
    series_dir
    for patient_dir in patient_dirs
    for series_dir in patient_dir.iterdir()
    if series_dir.is_dir()
    and series_dir.name.isdigit()
]

all_png_paths = [
    png_path
    for series_dir in series_dirs
    for png_path in series_dir.glob("*.png")
]

print("환자 폴더:", len(patient_dirs))
print("Series:", len(series_dirs))
print("PNG:", len(all_png_paths))

assert len(patient_dirs) == 413
assert len(series_dirs) == 2711
assert len(all_png_paths) == 121566

print("✅ 압축 해제본이 원본 감사 결과와 일치합니다.")

환자 폴더: 413
Series: 2711
PNG: 121566
✅ 압축 해제본이 원본 감사 결과와 일치합니다.


In [ ]:
#@title 저장되노 mainifest 복구
#모델 cohort 및 target 복구

series_view_manifest_df = pd.read_csv(
    AUDIT_DIR
    / "13_series_view_manifest.csv"
)

paper_model_series_df = pd.read_csv(
    AUDIT_DIR
    / "14_paper_model_series.csv"
)

extended_model_series_df = pd.read_csv(
    AUDIT_DIR
    / "15_extended_model_series.csv"
)

patient_label_summary_df = pd.read_csv(
    AUDIT_DIR
    / "19_patient_label_summary.csv"
)

paper_series_label_df = pd.read_csv(
    AUDIT_DIR
    / "20_paper_series_with_labels.csv"
)

extended_series_label_df = pd.read_csv(
    AUDIT_DIR
    / "21_extended_series_with_labels.csv"
)

patient_eligibility_df = pd.read_csv(
    AUDIT_DIR
    / "28_patient_modality_eligibility.csv"
)

print(
    "Paper series:",
    len(paper_series_label_df),
)

print(
    "Extended series:",
    len(extended_series_label_df),
)

print(
    "Paper 환자:",
    paper_series_label_df[
        "patient_id"
    ].nunique(),
)

print(
    "Multimodal 환자:",
    int(
        patient_eligibility_df[
            "paper_multimodal_available"
        ].sum()
    ),
)

assert len(paper_series_label_df) == 2676
assert len(extended_series_label_df) == 2680
assert (
    paper_series_label_df[
        "patient_id"
    ].nunique()
    == 412
)

print("✅ 모델 cohort를 복구했습니다.")

Paper series: 2676
Extended series: 2680
Paper 환자: 412
Multimodal 환자: 376
✅ 모델 cohort를 복구했습니다.


In [ ]:
#@title Clinical cleaning rule 재적용
#clinical 원본 및 규칙 불러오기

clinical_raw_df = pd.read_excel(
    FEATURES_FILE,
    sheet_name="Main",
)

clinical_raw_df.columns = [
    column.strip()
    if isinstance(column, str)
    else column
    for column in clinical_raw_df.columns
]

clinical_raw_df["ID"] = pd.to_numeric(
    clinical_raw_df["ID"],
    errors="raise",
).astype(int)

cleaning_rules_df = pd.read_csv(
    AUDIT_DIR
    / "36_clinical_cleaning_rules.csv"
)

print("Clinical 원본:", clinical_raw_df.shape)
print("Cleaning rules:", len(cleaning_rules_df))

assert clinical_raw_df.shape == (377, 60)
assert clinical_raw_df["ID"].is_unique

Clinical 원본: (377, 60)
Cleaning rules: 11


In [ ]:
#@title 원본 값 검증 후 규칙 적용

clinical_clean_df = clinical_raw_df.copy()

for rule in cleaning_rules_df.itertuples(
    index=False
):
    patient_id = int(rule.patient_id)
    column = rule.column
    expected_raw = rule.raw_value
    clean_value = rule.clean_value
    action = rule.action

    patient_mask = (
        clinical_clean_df["ID"]
        == patient_id
    )

    assert patient_mask.sum() == 1, (
        f"Patient {patient_id}가 "
        f"정확히 한 행이 아닙니다."
    )

    current_value = clinical_clean_df.loc[
        patient_mask,
        column,
    ].iloc[0]

    if pd.isna(expected_raw):
        assert pd.isna(current_value)
    else:
        assert np.isclose(
            float(current_value),
            float(expected_raw),
            rtol=1e-8,
            atol=1e-10,
        ), (
            f"원본 불일치: "
            f"patient={patient_id}, "
            f"column={column}, "
            f"expected={expected_raw}, "
            f"actual={current_value}"
        )

    if action != "KEEP":
        clinical_clean_df.loc[
            patient_mask,
            column,
        ] = clean_value

print("✅ 모든 cleaning rule이 원본 값과 일치했습니다.")

✅ 모든 cleaning rule이 원본 값과 일치했습니다.


In [ ]:
#@title LVEF 파생 열 추가

def calculate_lvef_grade(value):
    if pd.isna(value):
        return np.nan

    value = float(value)

    if value < 0.30:
        return 1

    if value <= 0.40:
        return 2

    if value < 0.55:
        return 3

    return 4


clinical_clean_df[
    "LVEF dysfunction derived"
] = clinical_clean_df["LVEF"].map(
    calculate_lvef_grade
)

clinical_clean_df[
    "LVEF dysfunction mismatch flag"
] = (
    clinical_clean_df[
        "LVEF dysfunction"
    ].notna()
    & clinical_clean_df[
        "LVEF dysfunction derived"
    ].notna()
    & (
        clinical_clean_df[
            "LVEF dysfunction"
        ]
        != clinical_clean_df[
            "LVEF dysfunction derived"
        ]
    )
).astype(int)

print(
    "원본 LVEF 등급 불일치:",
    int(
        clinical_clean_df[
            "LVEF dysfunction mismatch flag"
        ].sum()
    ),
)

원본 LVEF 등급 불일치: 41


In [ ]:
import pandas as pd
import numpy as np

STENOSIS_COLUMNS = [
    "LM",
    "Prox LAD",
    "Mid LAD",
    "Dist LAD",
    "1st dig",
    "2nd dig",
    "Prox LCX",
    "Mid LCX",
    "Dist LCX",
    "OM",
    "Prox RCA",
    "Mid RCA",
    "Dist RCA",
    "PDA",
    "PLB",
]

ALLOWED_STENOSIS_VALUES = {
    "NL",
    "1-25",
    "26-50",
    "51-75",
    "76-90",
    "91-99",
    "100",
}


def normalize_stenosis_value(value):
    if pd.isna(value):
        return pd.NA

    # Excel 숫자형 처리
    if isinstance(value, (int, np.integer)):
        return str(int(value))

    if isinstance(value, (float, np.floating)):
        if float(value).is_integer():
            return str(int(value))

    value = str(value).strip()

    # 여러 종류의 dash 통일
    value = (
        value.replace("–", "-")
        .replace("—", "-")
        .replace("−", "-")
    )

    if value.upper() == "NL":
        return "NL"

    if value == "100.0":
        return "100"

    return value


def normalize_stenosis_columns(df):
    result = df.copy()

    existing_columns = [
        column
        for column in STENOSIS_COLUMNS
        if column in result.columns
    ]

    for column in existing_columns:
        result[column] = (
            result[column]
            .map(normalize_stenosis_value)
            .astype("string")
        )

    return result


# ---------------------------------------------------------
# 양쪽 DataFrame에 동일한 정규화 적용
# ---------------------------------------------------------
clinical_clean_sorted_df = normalize_stenosis_columns(
    clinical_clean_sorted_df
)

previous_preview_sorted_df = normalize_stenosis_columns(
    previous_preview_sorted_df
)

# ---------------------------------------------------------
# 허용되지 않은 라벨 검사
# patient_id 열이 없어도 작동하도록 구성
# ---------------------------------------------------------
invalid_records = []

for column in STENOSIS_COLUMNS:
    if column not in clinical_clean_sorted_df.columns:
        continue

    invalid_mask = (
        clinical_clean_sorted_df[column].notna()
        & ~clinical_clean_sorted_df[column].isin(
            ALLOWED_STENOSIS_VALUES
        )
    )

    invalid_positions = np.flatnonzero(
        invalid_mask.to_numpy()
    )

    for position in invalid_positions:
        invalid_records.append({
            "row_position": int(position),
            "artery": column,
            "invalid_value": (
                clinical_clean_sorted_df.iloc[position][column]
            ),
        })

invalid_labels_df = pd.DataFrame(invalid_records)

print("허용되지 않은 stenosis 라벨:", len(invalid_labels_df))

if not invalid_labels_df.empty:
    display(invalid_labels_df)

assert invalid_labels_df.empty, (
    "허용되지 않은 stenosis 라벨이 있습니다."
)

# ---------------------------------------------------------
# 전처리 결과 재현성 비교
# ---------------------------------------------------------
pd.testing.assert_frame_equal(
    clinical_clean_sorted_df,
    previous_preview_sorted_df,
    check_dtype=False,
    check_exact=False,
    rtol=1e-10,
    atol=1e-12,
)

print("✅ Clinical 전처리 재현성 검증 통과")
print("✅ 숫자 100과 문자열 '100'을 표준 문자열로 통일했습니다.")

허용되지 않은 stenosis 라벨: 0
✅ Clinical 전처리 재현성 검증 통과
✅ 숫자 100과 문자열 '100'을 표준 문자열로 통일했습니다.


In [ ]:
PREVIEW_PATH = (
    AUDIT_DIR
    / "37_clinical_cleaned_preview.csv"
)

# ---------------------------------------------------------
# 이전 저장 결과 불러오기
# ---------------------------------------------------------
previous_preview_df = pd.read_csv(PREVIEW_PATH)

previous_preview_df.columns = [
    column.strip()
    if isinstance(column, str)
    else column
    for column in previous_preview_df.columns
]

# ---------------------------------------------------------
# 현재 결과와 이전 저장 결과 모두 같은 규칙으로 정규화
# ---------------------------------------------------------
clinical_clean_df = normalize_stenosis_columns(
    clinical_clean_df
)

previous_preview_df = normalize_stenosis_columns(
    previous_preview_df
)

# ---------------------------------------------------------
# 동일한 ID 순서로 정렬
# ---------------------------------------------------------
clinical_clean_sorted_df = (
    clinical_clean_df
    .sort_values("ID")
    .reset_index(drop=True)
)

previous_preview_sorted_df = (
    previous_preview_df
    .sort_values("ID")
    .reset_index(drop=True)
)

# ---------------------------------------------------------
# 재현성 검증
# ---------------------------------------------------------
pd.testing.assert_frame_equal(
    clinical_clean_sorted_df,
    previous_preview_sorted_df,
    check_dtype=False,
    check_exact=False,
    rtol=1e-10,
    atol=1e-10,
)

print("✅ Cleaning rule 재적용 결과가 이전 preview와 일치합니다.")
print("✅ Stenosis 라벨 자료형 차이도 정규화했습니다.")

✅ Cleaning rule 재적용 결과가 이전 preview와 일치합니다.
✅ Stenosis 라벨 자료형 차이도 정규화했습니다.


In [ ]:
clinical_clean_df.to_csv(
    PREVIEW_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("저장 완료:", PREVIEW_PATH)

저장 완료: /content/drive/MyDrive/[MacGyver]최종프로젝트/03_data/AngioCAD/audit_results/37_clinical_cleaned_preview.csv


In [ ]:
# Cleaning으로 실제 변경되거나 새로 생성된 열만 비교
rule_columns = sorted(
    cleaning_rules_df["column"]
    .unique()
    .tolist()
)

comparison_columns = [
    "ID",
    *rule_columns,
    "LVEF dysfunction derived",
    "LVEF dysfunction mismatch flag",
]

comparison_columns = [
    column
    for column in comparison_columns
    if (
        column in clinical_clean_sorted_df.columns
        and column in previous_preview_sorted_df.columns
    )
]

print("비교 대상 열:", comparison_columns)

pd.testing.assert_frame_equal(
    clinical_clean_sorted_df[
        comparison_columns
    ],
    previous_preview_sorted_df[
        comparison_columns
    ],
    check_dtype=False,
    check_exact=False,
    rtol=1e-10,
    atol=1e-10,
)

print(
    "✅ Cleaning 대상 열과 파생 열이 "
    "이전 preview와 일치합니다."
)

비교 대상 열: ['ID', 'BMI', 'BUN', 'Cr', 'Hgb', 'Length', 'MXD', 'Monocyte', 'Weight', 'LVEF dysfunction derived', 'LVEF dysfunction mismatch flag']
✅ Cleaning 대상 열과 파생 열이 이전 preview와 일치합니다.


In [ ]:
ARTERY_COLUMNS = [
    "LM",
    "Prox LAD",
    "Mid LAD",
    "Dist LAD",
    "1st dig",
    "2nd dig",
    "Prox LCX",
    "Mid LCX",
    "Dist LCX",
    "OM",
    "Prox RCA",
    "Mid RCA",
    "Dist RCA",
    "PDA",
    "PLB",
]


def normalize_label_for_compare(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    if text.endswith(".0"):
        text = text[:-2]

    return text


current_compare_df = (
    clinical_clean_sorted_df.copy()
)

previous_compare_df = (
    previous_preview_sorted_df.copy()
)

for column in ARTERY_COLUMNS:
    current_compare_df[column] = (
        current_compare_df[column]
        .map(normalize_label_for_compare)
    )

    previous_compare_df[column] = (
        previous_compare_df[column]
        .map(normalize_label_for_compare)
    )

common_columns = [
    column
    for column in current_compare_df.columns
    if column in previous_compare_df.columns
]

pd.testing.assert_frame_equal(
    current_compare_df[
        common_columns
    ],
    previous_compare_df[
        common_columns
    ],
    check_dtype=False,
    check_exact=False,
    rtol=1e-10,
    atol=1e-10,
)

print(
    "✅ 문자열/숫자 라벨을 정규화한 결과 "
    "전체 공통 열이 일치합니다."
)

✅ 문자열/숫자 라벨을 정규화한 결과 전체 공통 열이 일치합니다.


In [ ]:
added_columns = {
    "LVEF dysfunction derived",
    "LVEF dysfunction mismatch flag",
}

rule_column_set = set(rule_columns)

untouched_columns = [
    column
    for column in clinical_raw_df.columns
    if column not in rule_column_set
]

unexpected_changes = []

for column in untouched_columns:
    raw_values = clinical_raw_df[
        column
    ].reset_index(drop=True)

    clean_values = clinical_clean_df[
        column
    ].reset_index(drop=True)

    if pd.api.types.is_numeric_dtype(
        raw_values
    ):
        equal = np.allclose(
            pd.to_numeric(
                raw_values,
                errors="coerce",
            ),
            pd.to_numeric(
                clean_values,
                errors="coerce",
            ),
            equal_nan=True,
            rtol=1e-10,
            atol=1e-10,
        )
    else:
        raw_normalized = (
            raw_values
            .fillna("<MISSING>")
            .astype(str)
            .str.strip()
        )

        clean_normalized = (
            clean_values
            .fillna("<MISSING>")
            .astype(str)
            .str.strip()
        )

        equal = raw_normalized.equals(
            clean_normalized
        )

    if not equal:
        unexpected_changes.append(column)

print(
    "Cleaning rule 외 변경 열:",
    unexpected_changes,
)

assert not unexpected_changes

print(
    "✅ 지정된 cleaning rule 이외의 "
    "원본 열은 변경되지 않았습니다."
)

Cleaning rule 외 변경 열: []
✅ 지정된 cleaning rule 이외의 원본 열은 변경되지 않았습니다.


In [ ]:
#@title deterministic clinical 결과 저장

CLINICAL_CLEAN_PATH = (
    PREPROCESS_DIR
    / "01_clinical_deterministic_clean.csv"
)

clinical_clean_df.to_csv(
    CLINICAL_CLEAN_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("저장:", CLINICAL_CLEAN_PATH)

저장: /content/drive/MyDrive/[MacGyver]최종프로젝트/03_data/AngioCAD/preprocessing_results/01_clinical_deterministic_clean.csv


# 이미지 pixel hash 중복 검사@

In [ ]:
#@title pixel hash 함수

from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
import hashlib
import numpy as np


def compute_pixel_hash(path):
    path = Path(path)

    try:
        with Image.open(path) as image:
            grayscale = image.convert("L")
            array = np.asarray(grayscale)

        shape_bytes = (
            f"{array.shape[0]}x"
            f"{array.shape[1]}"
        ).encode("utf-8")

        pixel_hash = hashlib.sha256(
            shape_bytes + array.tobytes()
        ).hexdigest()

        return {
            "patient_id": int(
                path.parent.parent.name
            ),
            "series_id": int(
                path.parent.name
            ),
            "frame_name": path.name,
            "file_path": str(path),
            "height": int(array.shape[0]),
            "width": int(array.shape[1]),
            "pixel_sha256": pixel_hash,
            "status": "OK",
            "error": "",
        }

    except Exception as error:
        return {
            "patient_id": int(
                path.parent.parent.name
            ),
            "series_id": int(
                path.parent.name
            ),
            "frame_name": path.name,
            "file_path": str(path),
            "height": None,
            "width": None,
            "pixel_sha256": None,
            "status": "ERROR",
            "error": repr(error),
        }

In [ ]:
#@title 전체 PNG hash 계산

hash_records = []

with ThreadPoolExecutor(
    max_workers=8
) as executor:

    results = executor.map(
        compute_pixel_hash,
        all_png_paths,
    )

    for result in tqdm(
        results,
        total=len(all_png_paths),
        desc="Pixel hash 계산",
    ):
        hash_records.append(result)

image_hash_df = pd.DataFrame(
    hash_records
)

print(
    "전체 검사:",
    len(image_hash_df),
)

print(
    "Hash 성공:",
    int(
        (
            image_hash_df["status"]
            == "OK"
        ).sum()
    ),
)

print(
    "Hash 실패:",
    int(
        (
            image_hash_df["status"]
            == "ERROR"
        ).sum()
    ),
)

assert len(image_hash_df) == 121566
assert (
    image_hash_df["status"]
    == "OK"
).all()

print("✅ 전체 PNG pixel hash 계산을 완료했습니다.")

Pixel hash 계산:   0%|          | 0/121566 [00:00<?, ?it/s]

전체 검사: 121566
Hash 성공: 121566
Hash 실패: 0
✅ 전체 PNG pixel hash 계산을 완료했습니다.


In [ ]:
import pandas as pd
import numpy as np
from PIL import Image
from IPython.display import display

# ---------------------------------------------------------
# pixel hash 결과 DataFrame 자동 탐색
# ---------------------------------------------------------
required_columns = {
    "patient_id",
    "series_id",
    "frame_name",
    "file_path",
    "pixel_sha256",
}

hash_df = None
hash_df_name = None

for name, obj in list(globals().items()):
    if (
        isinstance(obj, pd.DataFrame)
        and required_columns.issubset(obj.columns)
    ):
        hash_df = obj.copy()
        hash_df_name = name
        break

if hash_df is None:
    raise RuntimeError(
        "pixel hash 결과 DataFrame을 찾지 못했습니다. "
        "hash 검사 셀부터 다시 실행해주세요."
    )

print("사용한 DataFrame:", hash_df_name)
print("전체 hash 검사 행:", len(hash_df))

hash_df["patient_id"] = pd.to_numeric(
    hash_df["patient_id"], errors="raise"
).astype(int)

hash_df["series_id"] = pd.to_numeric(
    hash_df["series_id"], errors="raise"
).astype(int)

# ---------------------------------------------------------
# 서로 다른 환자에서 발견된 동일 hash 탐색
# ---------------------------------------------------------
hash_group_summary = (
    hash_df.groupby("pixel_sha256", as_index=False)
    .agg(
        file_count=("file_path", "size"),
        patient_count=("patient_id", "nunique"),
        series_count=("series_id", "size"),
        patient_ids=(
            "patient_id",
            lambda x: ",".join(map(str, sorted(set(x)))),
        ),
        series_keys=(
            "file_path",
            lambda x: ",".join(
                sorted({
                    f"{p}/{s}"
                    for p, s in zip(
                        hash_df.loc[x.index, "patient_id"],
                        hash_df.loc[x.index, "series_id"],
                    )
                })
            ),
        ),
    )
)

cross_patient_groups = hash_group_summary.loc[
    hash_group_summary["patient_count"] > 1
].copy()

print("Cross-patient 중복 그룹:", len(cross_patient_groups))
display(cross_patient_groups)

cross_hashes = set(cross_patient_groups["pixel_sha256"])

cross_duplicate_rows = (
    hash_df.loc[hash_df["pixel_sha256"].isin(cross_hashes)]
    .sort_values(
        ["pixel_sha256", "patient_id", "series_id", "frame_name"]
    )
    .reset_index(drop=True)
)

duplicate_series_summary = (
    cross_duplicate_rows.groupby(
        ["patient_id", "series_id", "height", "width"],
        as_index=False,
    )
    .agg(
        duplicate_frame_count=("file_path", "size"),
        first_frame=("frame_name", "min"),
        last_frame=("frame_name", "max"),
        unique_hash_count=("pixel_sha256", "nunique"),
    )
)

display(duplicate_series_summary)

사용한 DataFrame: image_hash_df
전체 hash 검사 행: 121566
Cross-patient 중복 그룹: 1


,pixel_sha256,file_count,patient_count,series_count,patient_ids,series_keys
90520,beccf0ad0f1c882ad94f3b3f65967630d112cea73ba2b8...,129,3,129,"44,301,326","301/10,326/7,44/3"


,patient_id,series_id,height,width,duplicate_frame_count,first_frame,last_frame,unique_hash_count
0,44,3,512,1,1,frame_0000.png,frame_0000.png,1
1,301,10,512,1,127,frame_0000.png,frame_0511.png,1
2,326,7,512,1,1,frame_0511.png,frame_0511.png,1


In [ ]:
#@title 중복 그룹 분류

hash_count_df = (
    image_hash_df[
        "pixel_sha256"
    ]
    .value_counts()
    .rename_axis("pixel_sha256")
    .reset_index(name="file_count")
)

duplicate_hashes = set(
    hash_count_df.loc[
        hash_count_df["file_count"] > 1,
        "pixel_sha256",
    ]
)

duplicate_files_df = (
    image_hash_df[
        image_hash_df[
            "pixel_sha256"
        ].isin(duplicate_hashes)
    ]
    .copy()
)

duplicate_files_df[
    "series_key"
] = (
    duplicate_files_df[
        "patient_id"
    ].astype(str)
    + "/"
    + duplicate_files_df[
        "series_id"
    ].astype(str)
)

duplicate_group_records = []

for pixel_hash, group in (
    duplicate_files_df.groupby(
        "pixel_sha256"
    )
):
    patient_count = group[
        "patient_id"
    ].nunique()

    series_count = group[
        "series_key"
    ].nunique()

    if patient_count > 1:
        duplicate_scope = "CROSS_PATIENT"
    elif series_count > 1:
        duplicate_scope = (
            "SAME_PATIENT_CROSS_SERIES"
        )
    else:
        duplicate_scope = (
            "WITHIN_SAME_SERIES"
        )

    duplicate_group_records.append({
        "pixel_sha256": pixel_hash,
        "file_count": len(group),
        "patient_count": patient_count,
        "series_count": series_count,
        "duplicate_scope": duplicate_scope,
        "patient_ids": ",".join(
            map(
                str,
                sorted(
                    group[
                        "patient_id"
                    ].unique()
                ),
            )
        ),
        "series_keys": ",".join(
            sorted(
                group[
                    "series_key"
                ].unique()
            )
        ),
    })

duplicate_groups_df = pd.DataFrame(
    duplicate_group_records
)

if not duplicate_groups_df.empty:
    duplicate_groups_df = (
        duplicate_groups_df.sort_values(
            [
                "duplicate_scope",
                "file_count",
            ],
            ascending=[True, False],
        )
        .reset_index(drop=True)
    )

print("중복 pixel 그룹:", len(duplicate_groups_df))

if duplicate_groups_df.empty:
    print("중복 픽셀이 없습니다.")
else:
    display(
        duplicate_groups_df[
            "duplicate_scope"
        ].value_counts().to_frame(
            "group_count"
        )
    )

중복 pixel 그룹: 1


,group_count
duplicate_scope,
CROSS_PATIENT,1


In [ ]:
#@title cross-patient 중복 확인

cross_patient_groups_df = (
    duplicate_groups_df[
        duplicate_groups_df[
            "duplicate_scope"
        ] == "CROSS_PATIENT"
    ].copy()
    if not duplicate_groups_df.empty
    else pd.DataFrame()
)

print(
    "Cross-patient 중복 그룹:",
    len(cross_patient_groups_df),
)

if not cross_patient_groups_df.empty:
    display(
        cross_patient_groups_df.head(30)
    )

    cross_patient_hashes = set(
        cross_patient_groups_df[
            "pixel_sha256"
        ]
    )

    cross_patient_files_df = (
        duplicate_files_df[
            duplicate_files_df[
                "pixel_sha256"
            ].isin(
                cross_patient_hashes
            )
        ]
        .sort_values(
            [
                "pixel_sha256",
                "patient_id",
                "series_id",
                "frame_name",
            ]
        )
    )

    display(
        cross_patient_files_df.head(100)
    )
else:
    cross_patient_files_df = (
        pd.DataFrame(
            columns=duplicate_files_df.columns
        )
    )

    print(
        "✅ 서로 다른 환자 사이의 "
        "동일 픽셀 영상이 없습니다."
    )

Cross-patient 중복 그룹: 1


,pixel_sha256,file_count,patient_count,series_count,duplicate_scope,patient_ids,series_keys
0,beccf0ad0f1c882ad94f3b3f65967630d112cea73ba2b8...,129,3,3,CROSS_PATIENT,"44,301,326","301/10,326/7,44/3"


,patient_id,series_id,frame_name,file_path,height,width,pixel_sha256,status,error,series_key
13436,44,3,frame_0000.png,/content/AngioCAD/AngioCAD_Dataset/44/3/frame_...,512,1,beccf0ad0f1c882ad94f3b3f65967630d112cea73ba2b8...,OK,,44/3
87448,301,10,frame_0000.png,/content/AngioCAD/AngioCAD_Dataset/301/10/fram...,512,1,beccf0ad0f1c882ad94f3b3f65967630d112cea73ba2b8...,OK,,301/10
87738,301,10,frame_0001.png,/content/AngioCAD/AngioCAD_Dataset/301/10/fram...,512,1,beccf0ad0f1c882ad94f3b3f65967630d112cea73ba2b8...,OK,,301/10
87357,301,10,frame_0002.png,/content/AngioCAD/AngioCAD_Dataset/301/10/fram...,512,1,beccf0ad0f1c882ad94f3b3f65967630d112cea73ba2b8...,OK,,301/10
87554,301,10,frame_0003.png,/content/AngioCAD/AngioCAD_Dataset/301/10/fram...,512,1,beccf0ad0f1c882ad94f3b3f65967630d112cea73ba2b8...,OK,,301/10
...,...,...,...,...,...,...,...,...,...,...
87736,301,10,frame_0479.png,/content/AngioCAD/AngioCAD_Dataset/301/10/fram...,512,1,beccf0ad0f1c882ad94f3b3f65967630d112cea73ba2b8...,OK,,301/10
87365,301,10,frame_0480.png,/content/AngioCAD/AngioCAD_Dataset/301/10/fram...,512,1,beccf0ad0f1c882ad94f3b3f65967630d112cea73ba2b8...,OK,,301/10
87723,301,10,frame_0481.png,/content/AngioCAD/AngioCAD_Dataset/301/10/fram...,512,1,beccf0ad0f1c882ad94f3b3f65967630d112cea73ba2b8...,OK,,301/10
87383,301,10,frame_0482.png,/content/AngioCAD/AngioCAD_Dataset/301/10/fram...,512,1,beccf0ad0f1c882ad94f3b3f65967630d112cea73ba2b8...,OK,,301/10


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

PREPROCESSING_DIR = Path(
    "/content/drive/MyDrive/"
    "[MacGyver]최종프로젝트/"
    "03_data/AngioCAD/preprocessing_results"
)

PREPROCESSING_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

paper_series_df = pd.read_csv(
    AUDIT_DIR / "14_paper_model_series.csv"
)

extended_series_df = pd.read_csv(
    AUDIT_DIR / "15_extended_model_series.csv"
)

# 자료형 통일
for dataframe in [
    image_hash_df,
    paper_series_df,
    extended_series_df,
]:
    dataframe["patient_id"] = pd.to_numeric(
        dataframe["patient_id"],
        errors="raise",
    ).astype(int)

    dataframe["series_id"] = pd.to_numeric(
        dataframe["series_id"],
        errors="raise",
    ).astype(int)


def inspect_cohort_duplicates(
    image_df,
    cohort_series_df,
    cohort_name,
):
    cohort_keys = (
        cohort_series_df[
            ["patient_id", "series_id"]
        ]
        .drop_duplicates()
        .copy()
    )

    cohort_images = image_df.merge(
        cohort_keys,
        on=["patient_id", "series_id"],
        how="inner",
        validate="many_to_one",
    )

    cohort_images["series_key"] = (
        cohort_images["patient_id"].astype(str)
        + "/"
        + cohort_images["series_id"].astype(str)
    )

    hash_summary = (
        cohort_images.groupby(
            "pixel_sha256",
            as_index=False,
        )
        .agg(
            file_count=("file_path", "size"),
            patient_count=("patient_id", "nunique"),
            series_count=("series_key", "nunique"),
            patient_ids=(
                "patient_id",
                lambda values: ",".join(
                    map(
                        str,
                        sorted(set(values)),
                    )
                ),
            ),
            series_keys=(
                "series_key",
                lambda values: ",".join(
                    sorted(set(values))
                ),
            ),
        )
    )

    cross_patient_groups = hash_summary.loc[
        hash_summary["patient_count"] > 1
    ].copy()

    print(f"\n=== {cohort_name} cohort ===")
    print("검사 이미지:", len(cohort_images))
    print("검사 series:", len(cohort_keys))
    print(
        "Cross-patient 중복 그룹:",
        len(cross_patient_groups),
    )

    if cross_patient_groups.empty:
        print(
            "✅ 모델 cohort 내부 "
            "Cross-patient 중복이 없습니다."
        )
    else:
        print("❌ 추가 확인이 필요합니다.")
        display(cross_patient_groups)

    return cross_patient_groups


paper_cross_groups = inspect_cohort_duplicates(
    image_hash_df,
    paper_series_df,
    "Paper",
)

extended_cross_groups = inspect_cohort_duplicates(
    image_hash_df,
    extended_series_df,
    "Extended",
)


=== Paper cohort ===
검사 이미지: 119020
검사 series: 2676
Cross-patient 중복 그룹: 0
✅ 모델 cohort 내부 Cross-patient 중복이 없습니다.

=== Extended cohort ===
검사 이미지: 119046
검사 series: 2680
Cross-patient 중복 그룹: 0
✅ 모델 cohort 내부 Cross-patient 중복이 없습니다.


In [ ]:
paper_gate_pass = paper_cross_groups.empty
extended_gate_pass = extended_cross_groups.empty

gate1_result_df = pd.DataFrame([
    {
        "scope": "ALL_RAW_IMAGES",
        "cross_patient_duplicate_groups": 1,
        "status": "WARNING_QUARANTINED",
        "reason": (
            "중복이 제외된 1x512 비표준 "
            "series 3개에서만 발생"
        ),
    },
    {
        "scope": "PAPER_MODEL_COHORT",
        "cross_patient_duplicate_groups": (
            len(paper_cross_groups)
        ),
        "status": (
            "PASS" if paper_gate_pass else "FAIL"
        ),
        "reason": (
            "유효 cohort 내부 중복 없음"
            if paper_gate_pass
            else "추가 확인 필요"
        ),
    },
    {
        "scope": "EXTENDED_MODEL_COHORT",
        "cross_patient_duplicate_groups": (
            len(extended_cross_groups)
        ),
        "status": (
            "PASS"
            if extended_gate_pass
            else "FAIL"
        ),
        "reason": (
            "유효 cohort 내부 중복 없음"
            if extended_gate_pass
            else "추가 확인 필요"
        ),
    },
])

display(gate1_result_df)

gate1_result_df.to_csv(
    PREPROCESSING_DIR
    / "02_duplicate_gate1_decision.csv",
    index=False,
    encoding="utf-8-sig",
)

assert paper_gate_pass, (
    "Paper cohort 내부 중복을 확인해야 합니다."
)

assert extended_gate_pass, (
    "Extended cohort 내부 중복을 확인해야 합니다."
)

print("\n✅ 전처리 Gate 1 최종 통과")
print("✅ 환자 단위 fold 생성을 진행할 수 있습니다.")

,scope,cross_patient_duplicate_groups,status,reason
0,ALL_RAW_IMAGES,1,WARNING_QUARANTINED,중복이 제외된 1x512 비표준 series 3개에서만 발생
1,PAPER_MODEL_COHORT,0,PASS,유효 cohort 내부 중복 없음
2,EXTENDED_MODEL_COHORT,0,PASS,유효 cohort 내부 중복 없음



✅ 전처리 Gate 1 최종 통과
✅ 환자 단위 fold 생성을 진행할 수 있습니다.


In [ ]:
#@title hash 결과 저장

image_hash_df.to_csv(
    PREPROCESS_DIR
    / "02_image_pixel_hashes.csv",
    index=False,
    encoding="utf-8-sig",
)

duplicate_groups_df.to_csv(
    PREPROCESS_DIR
    / "03_duplicate_pixel_groups.csv",
    index=False,
    encoding="utf-8-sig",
)

duplicate_files_df.to_csv(
    PREPROCESS_DIR
    / "04_duplicate_pixel_files.csv",
    index=False,
    encoding="utf-8-sig",
)

cross_patient_files_df.to_csv(
    PREPROCESS_DIR
    / "05_cross_patient_duplicate_files.csv",
    index=False,
    encoding="utf-8-sig",
)

print("✅ 이미지 중복 감사 결과를 저장했습니다.")

✅ 이미지 중복 감사 결과를 저장했습니다.


In [ ]:
# =========================================================
# 전처리 Gate 1 최종 판정
# 전체 원본 경고와 모델 cohort 통과 여부를 분리
# =========================================================

global_duplicate_count = len(cross_patient_groups)
paper_duplicate_count = len(paper_cross_groups)
extended_duplicate_count = len(extended_cross_groups)

print("=== 전처리 Gate 1 최종 판정 ===")

# 전체 원본 결과는 경고로만 기록
if global_duplicate_count > 0:
    print(
        f"⚠️ 전체 원본 Cross-patient hash 그룹: "
        f"{global_duplicate_count}개"
    )
    print(
        "   제외된 1×512 비표준 series에서만 발생하여 "
        "격리 상태로 유지합니다."
    )
else:
    print("✅ 전체 원본 Cross-patient 중복 없음")

# 실제 모델 cohort를 Gate 기준으로 사용
print(
    f"Paper cohort Cross-patient 중복 그룹: "
    f"{paper_duplicate_count}"
)

print(
    f"Extended cohort Cross-patient 중복 그룹: "
    f"{extended_duplicate_count}"
)

gate1_pass = (
    paper_duplicate_count == 0
    and extended_duplicate_count == 0
)

if gate1_pass:
    print("\n✅ 전처리 Gate 1 최종 통과")
    print("✅ 환자 단위 fold 생성을 진행할 수 있습니다.")
else:
    print("\n❌ 모델 cohort 내부 중복이 남아 있습니다.")
    print("Cross-patient 중복 확인 전 fold를 만들지 않습니다.")

assert gate1_pass, (
    "Paper 또는 Extended 모델 cohort 내부에 "
    "Cross-patient 중복이 남아 있습니다."
)

=== 전처리 Gate 1 최종 판정 ===
⚠️ 전체 원본 Cross-patient hash 그룹: 1개
   제외된 1×512 비표준 series에서만 발생하여 격리 상태로 유지합니다.
Paper cohort Cross-patient 중복 그룹: 0
Extended cohort Cross-patient 중복 그룹: 0

✅ 전처리 Gate 1 최종 통과
✅ 환자 단위 fold 생성을 진행할 수 있습니다.
